# rl18_report

> Phase 18, Part B (`specs/multiagent/phase18/`): the analysis of the clone-RL runs on the colony and plain boards. Per run its mean over evaluation boards; H18-B1 (exact permutation test, colony-trained against plain-trained runs), H18-B2 (emergent division of labour: specialization index minus its shuffled null, sign-flip over runs), H18-B3 (gain over the warm start at N = 32), Holm over the three; H18-B4 (goal drift against the warm start, descriptive); a report and a figure; `syntheticRows18B` for tests.

In [ ]:
#| default_exp rl18_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Run values and tests

In [ ]:
#| export
import numpy as np

from explRL.bench_stats import holm
from explRL.rl16_report import permTest, signFlip
from explRL.rl18_grid import RL18_PLAN, RL18_RUNS

NAN = float('nan')


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return NAN


def runValue(rows, model, test, metric):
    "Mean of metric over a model's evaluation boards of one test (level '<model>|<test>')."
    v = [_f(r[metric]) for r in rows if r['level'] == f'{model}|{test}']
    v = [x for x in v if np.isfinite(x)]
    return float(np.mean(v)) if v else NAN


def trainSpec(turnsDir, nNull=50):
    """The training rollouts' specialization per step, from a run's turns/step-*.jsonl (Phase 17's saveTurns: one line per
    turn with N, nLiar, board and agent): {step: mean over the step's boards of specIndex - specNull} (policy slots only)."""
    import json
    from pathlib import Path
    from .colony_trace import specIndex
    out = {}
    for f in sorted(Path(turnsDir).glob('step-*.jsonl')):
        boards = {}
        for line in open(f):
            x = json.loads(line)
            if x.get('kind', 'turn') == 'turn':
                g = boards.setdefault((x['N'], x['nLiar'], x['board']), [[] for _ in range(x['N'])])
                g[x['agent']].append(x)
        d = [si - sn for (N, nL, _), slots in boards.items() for si, sn in [specIndex(slots, nL, nNull=nNull)] if np.isfinite(si - sn)]
        if d:
            out[int(f.stem.split('-')[1])] = float(np.mean(d))
    return out


def trendSlope(curve):
    "Least-squares slope of {step: value} per step (NaN with fewer than two steps)."
    s = sorted(curve)
    return float(np.polyfit(s, [curve[k] for k in s], 1)[0]) if len(s) > 1 else NAN


def runNames(arm, runs=RL18_RUNS):
    return [f'{a}_s{s}' for a, s in runs if a == arm]


def _boot(x, nboot=2000, seed=0):
    x = np.array([v for v in x if np.isfinite(v)], float)
    if not len(x):
        return [NAN, NAN]
    m = x[np.random.default_rng(seed).integers(0, len(x), (nboot, len(x)))].mean(1)
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]


def rl18Summaries(rows, plan=RL18_PLAN, nboot=2000, runs=RL18_RUNS, trainCurves=None):
    """Part B lite (RL18_PLAN v2): H18-B2 primary (Holm over plan['holm'], here one test), H18-B3 and H18-B4 secondary;
    H18-B1 only when p18team runs are present (a full Part B). Per model, honest pooled share on each evaluation test.
    trainCurves: {run: trainSpec curve} -> per run its first and last step's value and the slope (the secondary trend)."""
    C, P = runNames('c18team', runs), runNames('p18team', runs)
    rv = lambda ms, test, k: [runValue(rows, m, test, k) for m in ms]
    spec = [runValue(rows, m, 'colonyn8liar', 'specIndex') - runValue(rows, m, 'colonyn8liar', 'specNull') for m in C]
    b2 = signFlip(spec)
    warm32 = runValue(rows, 'warm', 'colonyn32liar', 'honestPooled')
    g32 = [v - warm32 for v in rv(C, 'colonyn32liar', 'honestPooled')]
    b3 = signFlip(g32)
    hyp = dict(H18_B2=dict(estimate=b2['mean'], p=b2['p'], n=len(spec), ci=_boot(spec, nboot)),
               H18_B3=dict(estimate=b3['mean'], p=b3['p'], n=len(g32), ci=_boot(g32, nboot), secondary=True))
    if P:
        b1 = permTest(rv(C, 'colonyn8liar', 'honestPooled'), rv(P, 'plainn8liar', 'honestPooled'))
        hyp['H18_B1'] = dict(estimate=b1['diff'], p=b1['p'], n=[len(C), len(P)])
    for h, a in zip(plan['holm'], holm([hyp[h]['p'] for h in plan['holm']])):
        hyp[h].update(pHolm=float(a), supported=bool(a < plan['alpha'] and hyp[h]['estimate'] > 0))
    drift = {}
    for arm, ms, test in (('c18team', C, 'colonyn8liar'), ('p18team', P, 'plainn8liar')):
        for k in ('freeRiding', 'misreport'):
            if not ms:
                continue
            d = [v - runValue(rows, 'warm', test, k) for v in rv(ms, test, k)]
            drift[f'{arm}|{k}'] = dict(estimate=float(np.nanmean(d)) if d else NAN, ci=_boot(d, nboot))
    hyp['H18_B4'] = dict(drift=drift, secondary=True)
    curves = {m: {t: runValue(rows, m, t, 'honestPooled') for t in sorted({r['level'].split('|', 1)[1] for r in rows})}
              for m in ['warm'] + C + P}
    tt = {m: dict(first=c[min(c)], last=c[max(c)], slope=trendSlope(c), curve=c) for m, c in (trainCurves or {}).items() if c}
    return dict(hyp=hyp, curves=curves, spec=dict(zip(C, spec)), gain32=dict(zip(C, g32)), trainTrend=tt)

## Report and figure

In [ ]:
#| export
def _fmt(x):
    return 'n/a' if x is None or not np.isfinite(x) else f'{x:.3f}'


def writeRL18(path, S, meta):
    "report.md: H18-B1 to B3 (estimate, exact p, Holm), H18-B4 (goal drift), per-run values."
    L = ['# Phase 18 Part B: clones trained with team reward on the colony board (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| hypothesis | estimate | p (one-sided, exact) | Holm | supported |', '|---|---|---|---|---|']
    for h in ('H18_B2', 'H18_B3', 'H18_B1'):
        if h in S['hyp']:
            t = S['hyp'][h]
            L.append(f"| {h}{' (secondary)' if t.get('secondary') else ''} | {_fmt(t['estimate'])} | {_fmt(t['p'])} | "
                     f"{_fmt(t.get('pHolm', float('nan')))} | {t.get('supported', 'n/a')} |")
    L += ['', '## H18-B4: goal drift against the warm start (N 8, with liars)', '', '| arm and measure | change | 95% interval |', '|---|---|---|']
    L += [f"| {k} | {_fmt(v['estimate'])} | [{_fmt(v['ci'][0])}, {_fmt(v['ci'][1])}] |" for k, v in S['hyp']['H18_B4']['drift'].items()]
    tr = S.get('trainTrend', {})
    L += ['', '## Per run', '', '| run | specialization minus null (N 8, liars) | gain at N 32 | training specialization: first, last step, slope |',
          '|---|---|---|---|']
    L += [f"| {m} | {_fmt(S['spec'][m])} | {_fmt(S['gain32'][m])} | " + (
          f"{_fmt(tr[m]['first'])}, {_fmt(tr[m]['last'])}, {_fmt(tr[m]['slope'])}" if m in tr else 'n/a') + ' |' for m in S['spec']]
    open(path, 'w').write('\n'.join(L) + '\n')


def figRL18(out, S):
    "rl18.png: honest pooled share with liars against N on the colony board, the warm start and each run."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.8))
    for m, d in S['curves'].items():
        Ns = sorted(int(t[7:-4]) for t in d if t.startswith('colonyn') and t.endswith('liar'))
        if Ns:
            ax.plot(Ns, [d[f'colonyn{N}liar'] for N in Ns], 'k-o' if m == 'warm' else '-', lw=2 if m == 'warm' else 0.8,
                    color=None if m == 'warm' else ('#1f6391' if m.startswith('c18') else '#c0392b'), label=m if m == 'warm' else None)
    ax.set(xscale='log', xlabel='team size', ylabel='honest pooled share (liars)', title='Colony board: trained clones (blue: colony-trained)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{out}/rl18.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

## Synthetic rows

In [ ]:
#| export
def syntheticRows18B(gain=0.05, spec=1.0, seed=0, boards=4, runs=RL18_RUNS, full=False):
    """Evaluation rows for the warm start and the runs with known effects (c18team: gain over warm and over p18team,
    specialization); the lite suite (colony board, N 8 and 32, liars) or, with full=True, every board, size and setting."""
    rng, rows = np.random.default_rng(seed), []
    models = ['warm'] + [f'{a}_s{s}' for a, s in runs]
    for m in models:
        g = gain if m.startswith('c18') else 0.0
        for b in ('colony', 'plain') if full else ('colony',):
            for N in (4, 8, 16, 32) if full else (8, 32):
                for liar in ('', 'liar') if full else ('liar',):
                    for _ in range(boards):
                        e = rng.normal(0, 0.01)
                        rows.append(dict(level=f'{m}|{b}n{N}{liar}', honestPooled=0.3 + g + e, specIndex=1.0 + (spec if m.startswith('c18') else 0) + e,
                                         specNull=1.0, freeRiding=0.1 + (0.05 if m != 'warm' else 0), misreport=1.0))
    return rows

## Tests

In [ ]:
from explRL.rl18_report import *
S = rl18Summaries(syntheticRows18B(), nboot=200)
h = S['hyp']
assert 'H18_B1' not in h and h['H18_B3']['secondary'] and 'pHolm' not in h['H18_B3']          # lite: one primary
assert abs(h['H18_B2']['estimate'] - 1.0) < 0.02 and h['H18_B2']['p'] == 1 / 32 and h['H18_B2']['supported']
assert abs(h['H18_B3']['estimate'] - 0.05) < 0.01 and h['H18_B3']['p'] == 1 / 32
assert abs(h['H18_B4']['drift']['c18team|freeRiding']['estimate'] - 0.05) < 1e-9 and 'p18team|freeRiding' not in h['H18_B4']['drift']
S0 = rl18Summaries(syntheticRows18B(gain=0.0, spec=0.0), nboot=200)
assert not S0['hyp']['H18_B2']['supported']
# a full Part B (p18team runs present): H18-B1 by permutation, Holm over three
from explRL.rl18_grid import RL18_PLAN as _P
full = [(a, s) for s in range(1, 9) for a in ('c18team', 'p18team') if s <= dict(c18team=8, p18team=5)[a]]
F = rl18Summaries(syntheticRows18B(runs=full, full=True), dict(_P, holm=['H18_B1', 'H18_B2', 'H18_B3']), 200, runs=full)
assert F['hyp']['H18_B1']['p'] == 1 / 1287 and F['hyp']['H18_B2']['p'] == 1 / 256 and all(F['hyp'][k]['supported'] for k in ('H18_B1', 'H18_B2', 'H18_B3'))
# the training rollouts' specialization: fixed roles (two slots share every turn, two never) score above a mixed board
import json, tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as d:
    for step, fixed in ((1, False), (2, True)):
        lines = [dict(N=4, nLiar=0, board=0, agent=a, kind='turn', action=['share' if (a < 2 if fixed else (a + t) % 2) else 'wait', None])
                 for a in range(4) for t in range(40)]
        Path(d, f'step-{step:04d}.jsonl').write_text(''.join(json.dumps(x) + '\n' for x in lines))
    cv = trainSpec(d, nNull=20)
assert cv[2] > cv[1] and trendSlope(cv) > 0
St = rl18Summaries(syntheticRows18B(), nboot=50, trainCurves={'c18team_s1': cv})
assert St['trainTrend']['c18team_s1']['last'] == cv[2]
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    writeRL18(f'{d}/report.md', S, dict(mode='test'))
    figRL18(d, S)
    assert os.path.getsize(f'{d}/rl18.png') > 0 and 'H18_B2' in open(f'{d}/report.md').read()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()